
Exercises XP

Last Updated: July 14th, 2025

👩‍🏫 👩🏿‍🏫 What You’ll learn

    Vector search strategies (KNN, ANN) and evaluation.
    Vector database utility (similarity search, RAG).
    Differences between vector databases, libraries, and plugins.
    Best practices for vector store usage and performance.
    How language models learn knowledge via context.
    Text embedding generation and vector storage.
    Querying vector stores for relevant documents.
    Applying language models for question answering with retrieved context.


🛠️ What you will create

A functional Retrieval-Augmented Generation (RAG) pipeline, demonstrating text vectorization, vector storage in FAISS and ChromaDB, similarity search, and question answering using a Hugging Face language model.


🌟 Exercise 1 : Data Loading and Preparation

In this exercise, we will set up the environment and prepare the dataset that we will use throughout this project. Proper data preparation ensures smooth downstream processes, such as generating embeddings, working with vector databases, or building machine learning models. Let’s walk through each step together.


Why This Step Matters:

Before diving into advanced techniques, it’s crucial to:

    Ensure all required libraries are installed.
    Load and inspect the data to understand its structure.
    Prepare a manageable subset for quicker iterations during development.

These steps help us avoid technical issues and ensure our analysis or models are built on a solid foundation.


Instructions:

1. Install Required Libraries

The project requires specialized libraries for vector search and database management:

Enter your folder, then, in your terminal :


pip install -q faiss-cpu==1.7.4 
pip install -q chromadb==0.3.21
pip install -qU chromadb
pip install -q numpy<2


Create a Cache Directory to keep our workspace organized and handle any intermediate data or downloaded files:

In your terminal :

mkdir cache

then

apt install libomp-dev
python -m pip install --upgrade faiss-cpu


then, in your file, import essential libraries

import numpy as np
import pandas as pd
import faiss
import json
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


    faiss-cpu: A library for efficient similarity search and clustering of dense vectors (developed by Facebook AI Research).
    chromadb: A vector database library that allows us to store and query embeddings efficiently.

These libraries are essential for later stages when we handle embeddings and perform similarity searches.

2. Load the Dataset

We’ll be working with a dataset called labelled_newscatcher_dataset.csv, which contains labeled news articles. These articles will later be processed into embeddings for vector storage and search.

    Task: Load the dataset into a pandas DataFrame:


path =  # Provide the correct file path.
pdf =   # Load the CSV file into a pandas DataFrame.


This step ensures that our data is in a format suitable for analysis.

3. Add an Identifier Column (if needed)

Unique identifiers help us track each record, especially when we work with vector databases:


pdf["id"] =


Each news article will have a unique ID, making it easier to reference during storage and retrieval.

4. Inspect the Data

Use the following command to get a quick overview of the dataset:


display(pdf)



Take a moment to observe:

    The available columns.
    The type of data they contain (e.g., text, labels).
    Whether there are any missing values.

Understanding the dataset at this stage is critical for informed decision-making in subsequent steps.

5. Create a Subset for Faster Processing

Working with large datasets can be time-consuming. To enable faster iterations during development:

    Task: Select a smaller subset of the DataFrame (e.g., the first 1000 rows).

This approach lets you test your code efficiently before scaling up to the entire dataset.


🌟 Exercise 2: Vectorization with Sentence Transformers

In this exercise, we will transform our textual data (news titles) into numerical representations known as embeddings. This step is crucial for enabling machines to understand and work with text data in tasks like similarity search, clustering, and machine learning. We will use Sentence Transformers, a popular library for generating dense vector representations of text.


Why This Step Matters:

Machines cannot directly process raw text—they need numerical input. Embeddings are dense vectors that capture the meaning and context of text. By generating embeddings for our news titles, we make them usable for downstream tasks such as similarity searches or feeding into machine learning models.


Instructions:

1. Install and Import Sentence Transformers Library

The sentence_transformers library provides easy-to-use methods for generating sentence-level embeddings.


from sentence_transformers import InputExample


    InputExample: A utility class that helps format data inputs for training or inference with sentence transformers.

2. Prepare the Data for Embedding Generation

We will apply a helper function to the subset of our DataFrame that we created earlier. This function formats each row into an InputExample object, which is required for the embedding process.

    Task: Extract the subset of the DataFrame (e.g., pdf_subset) for which you want to generate embeddings.


pdf_subset = ...  # Use the subset created in the previous exercise.


3. Create a Helper Function

This function converts each record (news title) into the proper format (InputExample) required by the Sentence Transformer model.


def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Helper function that outputs a sentence_transformer guid, label, and text.
    """
    return ...  # Format and return the InputExample.


    The function will take a row (in this case, the title of a news article) and format it properly for the embedding generation process.

4. Apply the Helper Function to the Subset

We’ll apply this function across the subset DataFrame to generate a list of InputExample objects:


faiss_train_examples = pdf_subset.apply(lambda x: example_create_fn(x["title"]), axis=1).tolist()
faiss_train_examples[:10]


This prepares the data for embedding generation by converting each news title into a structured format.

5. Initialize the Embedding Model

We will use the pre-trained model all-MiniLM-L6-v2, which provides high-quality embeddings for a wide range of natural language processing (NLP) tasks.

    Task: Initialize the model.


from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    ...  # Specify the model name.
)


    This step loads the model into memory, ready for embedding generation.

6. Extract the Titles and Convert to a List of Strings

Extract the “title” column from your DataFrame subset and convert it into a list. This is the raw text data we’ll be embedding.

    Task: Convert the titles into a list of strings.


# Example (fill in appropriately):
titles_list = pdf_subset["title"].tolist()


7. Generate Embeddings for the Titles

Using the initialized model, generate embeddings for each title:


faiss_title_embedding =  # Generate embeddings for the list of titles.


    This step transforms each title into a dense vector that captures its semantic meaning.


8. Check Embedding Dimensions

To verify the embeddings were generated correctly, check the shape of the output:


len(faiss_title_embedding), len(faiss_title_embedding[0])


    This confirms how many embeddings you have (one per title) and the dimensionality of each embedding vector.


🌟 Exercise 3: FAISS Indexing and Search

In this exercise, we will use FAISS (Facebook AI Similarity Search) to build an index of the embeddings generated in the previous exercise. This allows us to perform fast and efficient similarity searches over large collections of vectors. The goal is to make it possible to retrieve the most relevant news articles based on a user’s query.


Why This Step Matters:

FAISS is a library designed to perform similarity search at scale. When working with embeddings (which are high-dimensional vectors), searching through them efficiently becomes challenging. FAISS provides optimized algorithms for indexing and searching, making it possible to retrieve similar items in milliseconds, even from large datasets.


Instructions:

    Install and Import FAISS Library
    If you haven’t already, ensure FAISS is installed and import the necessary modules:


import numpy as np
import faiss


    numpy: For handling arrays and matrix operations.
    faiss: To build and query the vector index.


2. Prepare the Data for Indexing

Use the embedding vectors generated from the previous exercise and prepare them for indexing:


pdf_to_index =  # This should be your subset DataFrame containing the articles.
id_index =  # An array of IDs corresponding to each article.


    pdf_to_index: The subset of the DataFrame that we want to index.
    id_index: An array of unique IDs for each embedding vector.


3. Normalize the Embedding Vectors

To perform cosine similarity search (which measures the angle between vectors rather than their distance), we first need to normalize the embedding vectors:


content_encoded_normalized =  # Embedding vectors.
faiss.normalize_L2(content_encoded_normalized)


    Normalization ensures that the vectors have unit length, which is necessary for cosine similarity to work correctly.


4. Create the FAISS Index

FAISS provides different types of indexes depending on the similarity measure and search requirements. We will use an IndexFlatIP (Inner Product) wrapped in an IndexIDMap:


index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
index_content.add_with_ids(content_encoded_normalized, id_index)


    IndexFlatIP: An index type that uses inner product (which is equivalent to cosine similarity for normalized vectors).
    IndexIDMap: Maps search results back to the original IDs, ensuring we can retrieve the corresponding articles.

This step builds the index and adds the normalized vectors along with their IDs.


5. Implement a Search Function

Next, we’ll define a function search_content that takes a user query and retrieves the most similar articles from the index:


def search_content(query, pdf_to_index, k=3):
    query_vector =  # Encode the query string into an embedding vector.
    faiss.normalize_L2(query_vector)  # Normalize the query vector.

    # Perform the search
    top_k =  # The top-k similar vectors.
    ids =  # The IDs of the matching vectors.
    similarities =  # Similarity scores for the matches.

    results =  # Retrieve the matching articles from pdf_to_index.
    results["similarities"] = similarities  # Add similarity scores.
    return results


    This function encodes the user’s query into a vector, searches the FAISS index, retrieves the top-k most similar vectors, and returns the matching articles along with their similarity scores.


6. Test the Search Function

Use the search function to find articles related to a sample query:


display(search_content("animal", pdf_to_index, k=5))


This allows you to verify that the search process works and returns relevant articles.


🌟 Exercise 4: ChromaDB Collection and Querying

In this exercise, we will introduce ChromaDB, an open-source vector database designed to store, index, and query embedding vectors. ChromaDB simplifies working with embeddings, and unlike FAISS, it can automatically handle tokenization, embedding, and indexing without requiring manual embedding generation. This makes it ideal for integrating with LLM-based applications (Large Language Model applications), especially in building Q&A systems or search engines.


Why This Step Matters:

With embeddings generated for our data, the next logical step is to store and query these embeddings efficiently. ChromaDB provides a higher-level interface for managing embeddings and supports metadata, making it a good fit for building applications like document search or Q&A systems. By using ChromaDB, we demonstrate how to integrate embeddings into a real-world workflow that supports querying and retrieving relevant documents.


Instructions:

1. Install and Import ChromaDB Library

Ensure you have ChromaDB installed and import the necessary components:


import chromadb
from chromadb.config import Settings


    chromadb: The main library for managing vector collections and queries.
    Settings: Configuration options for ChromaDB.


2. Initialize a ChromaDB Client and Create a Collection

ChromaDB organizes vectors into collections, which are similar to tables in a database. Each collection holds a set of documents (vectors) and associated metadata.


chroma_client = chromadb.Client()
collection_name = "my_news"

# If a collection with the same name exists, delete it to avoid conflicts
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)


    This code initializes the ChromaDB client, checks if a collection named “my_news” already exists, deletes it if it does, and creates a fresh collection.
    Collections store both documents (the text or embeddings) and metadata (e.g., topic labels).


3. Add Data to the Collection

ChromaDB simplifies data ingestion by automatically generating embeddings if you don’t supply a custom embedding model. It uses the default SentenceTransformerEmbeddingFunction, which handles tokenization, embedding, and indexing.

    Task: Add the first 100 news titles from the DataFrame subset to the collection. Alongside each title, include its corresponding topic as metadata and assign a unique ID for each document.


# Display the DataFrame subset (for reference)
display(pdf_subset)

collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": topic} for topic in pdf_subset["topic"][:100].tolist()],
    ids=...  # Provide a list of unique IDs.
)


    The documents parameter holds the list of news titles.
    The metadatas parameter holds the associated topics as metadata.
    The ids parameter must be a list of unique identifiers (e.g., strings or integers) for each document.

    Note: Adding data to the collection may take time depending on the volume of data, as ChromaDB processes and indexes the text behind the scenes.


4. Query the Collection

Finally, perform a search query to retrieve the most relevant documents based on a search term.

    Task: Query the collection using a term (e.g., “space”) and retrieve the top 10 most relevant documents.


import json

results = ...  # Perform the search query.

print(json.dumps(results, indent=4))


    The search term (e.g., “space”) is automatically converted into an embedding by ChromaDB, and the collection returns the 10 nearest neighbors—documents most semantically similar to the query.
    The results include the matched documents, their metadata, and similarity scores.


🌟 Exercise 5: Question Answering with Hugging Face Model

In this exercise, we will bring everything together by building a Question Answering (Q/A) system using a Hugging Face language model. By combining document retrieval (via ChromaDB) with text generation (via Hugging Face), we create a simple yet powerful pipeline where a model generates answers based on relevant context.


Why This Step Matters:

Retrieving relevant documents is only half the battle. The next step is generating meaningful responses based on that retrieved content. This is a core technique in modern Retrieval-Augmented Generation (RAG) systems, where a language model leverages both pre-trained knowledge and external information to answer questions more accurately. By integrating ChromaDB and Hugging Face transformers, we simulate a real-world Q/A pipeline.


Instructions:

1. Install and Import the Transformers Library

The Hugging Face transformers library provides access to a variety of pre-trained language models.


from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


    AutoTokenizer: Automatically loads the appropriate tokenizer for the selected model.
    AutoModelForCausalLM: Loads a causal language model (such as GPT-2) for text generation.
    pipeline: A high-level interface for common tasks like text generation.


2. Initialize the Model and Tokenizer

Select a pre-trained model for text generation (e.g., GPT-2 or a similar causal language model) and initialize both the model and its tokenizer:


model_id =  # Specify the Hugging Face model ID (e.g., 'gpt2').
tokenizer =  # Load the tokenizer for the model.
lm_model =  # Load the causal language model.


    The model generates text based on provided input.
    The tokenizer converts between raw text and the tokenized format needed by the model.

3. Create a Text Generation Pipeline

Set up a pipeline for text generation, which wraps the model and tokenizer into a convenient interface:


pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,  # Maximum number of tokens to generate.
    device_map="auto",   # Automatically uses available GPU/CPU resources.
)


    This simplifies running inference and generating text with the model.

4. Construct a Prompt Template

The prompt includes both the retrieved context (from ChromaDB) and the user’s question. This way, the model generates a response informed by the relevant documents.


question =  # Define the user's question (e.g., "What's the latest news on space development?").
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])  # Concatenate the retrieved documents.
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"


    Context: A concatenation of retrieved documents that provide background information.
    Question: The user’s query.
    Prompt: Combines both to guide the language model’s response.


5. Generate a Response Using the Pipeline

Feed the prompt to the text generation pipeline and generate a response:


lm_response =  # Use the pipeline to generate text based on the prompt.
print(lm_response[0]["generated_text"])


    The output is a generated text string that attempts to answer the user’s question using the provided context.


6. Experiment with Different Prompts and Context Windows

Try varying the question and the context size (e.g., using more or fewer retrieved documents) to observe how the model’s responses change.

In [10]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # force CPU for safety

import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import faiss


from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


print("Exercise 1 : Data Loading and Preparation")

# 1) Paths / install notes (already covered above)
PATH = "labelled_newscatcher_dataset.csv"  # change if needed

# 2) Load dataset
if not os.path.exists(PATH):
    raise FileNotFoundError(f"CSV not found at: {PATH}")
pdf = pd.read_csv(PATH, sep=";", engine="python", on_bad_lines="skip", encoding="utf-8")

# 3) Add ID column if needed
if "id" not in pdf.columns:
    pdf["id"] = np.arange(len(pdf))

# 4) Inspect data
print("\nDataframe head():")
print(pdf.head())
print("\nColumns:", pdf.columns.tolist())
print("Shape:", pdf.shape)

# 5) Create subset for faster processing (change N if needed)
N = 1000
pdf_subset = pdf.iloc[:N].reset_index(drop=True)
print(f"\nSubset created: {pdf_subset.shape} rows.")

# Basic sanity checks on columns we will use
# We will assume typical newscatcher columns: "title", "topic" (adjust if your CSV differs)
expected_cols = ["title", "topic"]
missing_cols = [c for c in expected_cols if c not in pdf_subset.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in CSV: {missing_cols}. Please adapt the code.")



print("Exercise 2 : Vectorization with Sentence Transformers")

# 1) Import done above (InputExample)
# 2) pdf_subset already created

# 3) Helper function to create InputExample (even if we won't fine-tune, we follow the exercise)
def example_create_fn(title: str) -> InputExample:
    """
    Helper: returns a sentence-transformers InputExample for a single title
    """
    return InputExample(texts=[str(title)], label=0.0)

faiss_train_examples = pdf_subset.apply(lambda x: example_create_fn(x["title"]), axis=1).tolist()
print("First 3 InputExample objects:")
print(faiss_train_examples[:3])

# 5) Initialize embedding model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
st_model = SentenceTransformer(model_name, device="cpu")

# 6) Titles list
titles_list = pdf_subset["title"].astype(str).tolist()

# 7) Generate embeddings
faiss_title_embedding = st_model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)

# 8) Check dimensions
print("\nEmbeddings shape:", faiss_title_embedding.shape)
print("Example vector dim:", faiss_title_embedding.shape[1])



print("Exercise 3 : FAISS Indexing and Search")

pdf_to_index = pdf_subset.copy()
id_index = pdf_to_index["id"].astype(np.int64).values

# 3) Normalize embeddings for cosine similarity via IP
content_encoded_normalized = faiss_title_embedding.astype(np.float32).copy()
faiss.normalize_L2(content_encoded_normalized)

# 4) Create index (Inner Product == Cosine when normalized)
dim = content_encoded_normalized.shape[1]
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
index_content.add_with_ids(content_encoded_normalized, id_index)
print("FAISS index built. Total vectors:", index_content.ntotal)

# 5) Search function
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3) -> pd.DataFrame:
    # Encode query
    q_vec = st_model.encode([query], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(q_vec)

    # Search
    similarities, ids = index_content.search(q_vec, k)
    ids = ids[0]
    sims = similarities[0]

    # Map IDs to rows
    hits = pdf_to_index[pdf_to_index["id"].isin(ids)].copy()

    # Reorder hits by similarity (descending)
    # create map id->sim
    sim_map = dict(zip(ids.tolist(), sims.tolist()))
    hits["similarities"] = hits["id"].map(sim_map)
    hits = hits.sort_values("similarities", ascending=False).reset_index(drop=True)
    return hits

print("\nFAISS: Example search for query='animal', k=5")
print(search_content("animal", pdf_to_index, k=5)[["id", "title", "topic", "similarities"]].head())



print("Exercise 4 : ChromaDB Collection and Querying")

DB_DIR = "cache/chroma_db"
os.makedirs(DB_DIR, exist_ok=True)

try:
    chroma_client = chromadb.PersistentClient(path=DB_DIR)
except AttributeError:
    from chromadb.config import Settings
    chroma_client = chromadb.Client(Settings(
        chroma_db_impl="duckdb+parquet",
        persist_directory=DB_DIR
    ))

collection_name = "my_news"

# Compat : list_collections() renvoie des objets avec un attribut .name
existing = [c.name for c in chroma_client.list_collections()]
if collection_name in existing:
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

# 3) Add first 100 titles with their topics (inchangé)
K = min(100, len(pdf_subset))
docs_to_add = pdf_subset["title"].iloc[:K].astype(str).tolist()
metas_to_add = [{"topic": t} for t in pdf_subset["topic"].iloc[:K].astype(str).tolist()]
ids_to_add = [str(i) for i in pdf_subset["id"].iloc[:K].tolist()]

collection.add(
    documents=docs_to_add,
    metadatas=metas_to_add,
    ids=ids_to_add
)

print(f"Inserted {K} docs into Chroma collection '{collection_name}'")

# 4) Query collection (inchangé)
query_term = "space"
results = collection.query(
    query_texts=[query_term],
    n_results=10
)
print("\nChromaDB query results (top 10) for:", query_term)
print(json.dumps(results, indent=2))


print("Exercise 5 : Question Answering with a Hugging Face model")

# 1) Imports already done
# 2) Initialize causal LM
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)

# 3) Pipeline (CPU force)
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device=-1  # CPU
)

# 4) Build prompt
question = "What's the latest news on space development?"
# Use the Chroma results above (first query) as context
context_docs = results["documents"][0] if results and "documents" in results and len(results["documents"]) > 0 else []
context = " ".join([f"#{str(i)}" for i in context_docs])
prompt_template = f"Relevant context: {context}\n\nThe user's question: {question}\n\nAnswer:"

# 5) Generate answer
lm_response = pipe(prompt_template, do_sample=True, top_p=0.9, temperature=0.8)
print("\n--- Generated Answer ---\n")
print(lm_response[0]["generated_text"])

Exercise 1 : Data Loading and Preparation

Dataframe head():
     topic                                               link          domain  \
0  SCIENCE  https://www.eurekalert.org/pub_releases/2020-0...  eurekalert.org   
1  SCIENCE  https://www.pulse.ng/news/world/an-irresistibl...        pulse.ng   
2  SCIENCE  https://www.express.co.uk/news/science/1322607...   express.co.uk   
3  SCIENCE  https://www.ndtv.com/world-news/glaciers-could...        ndtv.com   
4  SCIENCE  https://www.thesun.ie/tech/5742187/perseid-met...       thesun.ie   

        published_date                                              title  \
0  2020-08-06 13:59:45  A closer look at water-splitting's solar fuel ...   
1  2020-08-12 15:14:19  An irresistible scent makes locusts swarm, stu...   
2  2020-08-13 21:01:00  Artificial intelligence warning: AI will know ...   
3  2020-08-03 22:18:26   Glaciers Could Have Sculpted Mars Valleys: Study   
4  2020-08-12 19:54:36  Perseid meteor shower 2020: What time and h

Batches:   0%|          | 0/32 [00:00<?, ?it/s]


Embeddings shape: (1000, 384)
Example vector dim: 384
Exercise 3 : FAISS Indexing and Search
FAISS index built. Total vectors: 1000

FAISS: Example search for query='animal', k=5
    id                                              title       topic  \
0  176  Random: You Can Pick Up and Pet Cats in Assass...  TECHNOLOGY   
1  975  Researchers explore social behavior of animals...      HEALTH   
2   99              Ghostwire: Tokyo confirms dog petting  TECHNOLOGY   
3  928                 Just Let This Lizard Be a Dinosaur     SCIENCE   
4  762  'Secret' life of sharks: Study reveals their s...     SCIENCE   

   similarities  
0      0.391902  
1      0.376784  
2      0.344059  
3      0.317387  
4      0.295497  
Exercise 4 : ChromaDB Collection and Querying
Creating collection 'my_news'


C:\Users\mathi\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:07<00:00, 11.1MiB/s]


Inserted 100 docs into Chroma collection 'my_news'

ChromaDB query results (top 10) for: space
{
  "ids": [
    [
      "72",
      "7",
      "30",
      "26",
      "23",
      "76",
      "69",
      "40",
      "47",
      "75"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "Beck teams up with NASA and AI for 'Hyperspace' visual album experience",
      "Orbital space tourism set for rebirth in 2021",
      "NASA drops \"insensitive\" nicknames for cosmic objects",
      "\u2018It came alive:\u2019 NASA astronauts describe experiencing splashdown in SpaceX Dragon",
      "Hubble Uses Moon As \u201cMirror\u201d to Study Earth\u2019s Atmosphere \u2013 Proxy in Search of Potentially Habitable Planets Around Other Stars",
      "Australia's small yet crucial part in the mission to find life on Mars",
      "NASA Astronauts in SpaceX Capsule Splashdown in Gulf Of Mexico",
      "SpaceX's Starship spacecraft saw 150 meters high",
      "NASA\u2019s InSight lander shows wha

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- Generated Answer ---

Relevant context: #Beck teams up with NASA and AI for 'Hyperspace' visual album experience #Orbital space tourism set for rebirth in 2021 #NASA drops "insensitive" nicknames for cosmic objects #‘It came alive:’ NASA astronauts describe experiencing splashdown in SpaceX Dragon #Hubble Uses Moon As “Mirror” to Study Earth’s Atmosphere – Proxy in Search of Potentially Habitable Planets Around Other Stars #Australia's small yet crucial part in the mission to find life on Mars #NASA Astronauts in SpaceX Capsule Splashdown in Gulf Of Mexico #SpaceX's Starship spacecraft saw 150 meters high #NASA’s InSight lander shows what’s beneath Mars’ surface #Alien base on Mercury: ET hunters claim to find huge UFO

The user's question: What's the latest news on space development?

Answer: It is very exciting to be involved in this amazing project. It is also very exciting to be part of a team that will continue to support and develop the next generation of space exploration. 